In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
from torch.utils.data import TensorDataset
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

#nee of reshape
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print('X_train_tensor:', X_train_tensor.shape, X_train_tensor.dtype)
print('y_train_tensor:', y_train_tensor.shape, y_train_tensor.dtype)



In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

print('Train dataset size:', len(train_dataset))
print('Test dataset size :', len(test_dataset))


In [ ]:
# 3. Create DataLoaders
batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
#buha no need for shuffle


In [ ]:
# 4. Print shape of one batch
X_batch, y_batch = next(iter(train_loader))
print('X batch shape:', X_batch.shape)  # here we wil l get the dim sh w d
print('y batch shape:', y_batch.shape)  # (batch, 1) cuz of view



In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

X_batch, y_batch = next(iter(train_loader))

num_images = 6#7
plt.figure(figsize=(num_images * 3, 3))

for i in range(num_images):
    img = X_batch[i].permute(1, 2, 0).numpy()
    ax = plt.subplot(1, num_images, i + 1)
    ax.imshow(img)
    ax.set_title(f'Age: {y_batch[i].item():.0f}')
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:

import torch.nn as nn

class NN4(nn.Module):

    def __init__(self, input_dim: int, hidden_dim: int):
        super(NN4, self).__init__()

        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim, 1)  # one ouput cuz reg

        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: (batch_size, input_dim)
        a1 = self.relu(self.layer1(x))
        a2 = self.relu(self.layer2(a1))
        a3 = self.relu(self.layer3(a2))
        la = self.layer4(a3)
        return la


In [ ]:
# Task 2: Write your training loop here:

import torch

def train_one_epoch(model, optimizer, criterion, train_loader, device):
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.view(X_batch.size(0), -1).to(device)
        y_batch = y_batch.view(-1, 1).to(device)  # try to flatten last lab error


        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    return avg_loss


In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
    model.eval()

    running_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:

            X_batch = X_batch.view(X_batch.size(0), -1).to(device)
            y_batch = y_batch.view(-1, 1).to(device) #again avod last error

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

    avg_loss = running_loss / len(test_loader)
    return avg_loss


In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Compute flattened input dimension
input_dim =3 * 36* 36 #from previouse

model = NN4(input_dim=input_dim, hidden_dim=16).to(device)
print(model)

criterion = nn.MSELoss()  #  loss


optimizer = AdamW(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Start training for 20 epochs:

num_epochs = 20
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)
    val_loss = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

print('Training Complete!')


In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoooch')
plt.ylabel('MSE    Loss   ')
plt.title('Training & Validation Loss over Epochs')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here: #notime